# EVA-02: A Visual Representation Powerhouse Reconstructed from EVA

**Paper:** Y. Fang, W. Sun, X. Wang, L. Wu, J. Wang, D. Cao, X. Chang, and Y. Wang, "EVA-02: A Visual Representation Powerhouse Reconstructed from EVA," *arXiv preprint arXiv:2303.11331*, 2023. [arXiv:2303.11331](https://arxiv.org/abs/2303.11331)

**Summary:** EVA-02 reconstructs ViT using three targeted improvements — 2D Rotary Position Embeddings (RoPE), SwiGLU feed-forward networks, and Sub-Layer Normalization for Q and K — achieving state-of-the-art visual representations through CLIP-aligned pre-training.

## Architecture Overview

```
Input Image [B, 3, H, W]
        │
        ▼  PatchEmbed (Conv2d, patch_size=16 or 14)
Patches [B, N, D]  — NO positional embedding added here!
        │            Positions are encoded in Q,K via RoPE inside each block
        ▼  ┌────────────────────────────────────────────────────────────────┐
           │  EVABlock × depth                                              │
           │  ┌──────────────────────────────────────────────────────────┐  │
           │  │  x = x + EVAAttention(LN(x),  rope_cos, rope_sin)       │  │
           │  │  x = x + SwiGLUFFN(LN(x))                               │  │
           │  │                                                          │  │
           │  │  EVAAttention internals:                                 │  │
           │  │    Q,K,V  = in_proj(x)                                   │  │
           │  │    Q = q_norm(Q)  ← Sub-LN on Q                         │  │
           │  │    K = k_norm(K)  ← Sub-LN on K                         │  │
           │  │    Q,K = apply_rope(Q,K, rope_cos, rope_sin)            │  │
           │  │    Attn = softmax(QKᵀ/√d) · V                           │  │
           │  │                                                          │  │
           │  │  SwiGLUFFN internals:                                    │  │
           │  │    gate  = SiLU(W₁·x)  ← gated branch                   │  │
           │  │    value = W₂·x         ← value branch                  │  │
           │  │    out   = W₃·(gate ⊙ value)                            │  │
           │  │    hidden = ⌈8/3 × D / 256⌉ × 256  (≈2.67D)            │  │
           │  └──────────────────────────────────────────────────────────┘  │
           └────────────────────────────────────────────────────────────────┘
        │
        ▼  Final LayerNorm
        │
        ├──► pooled_output = mean(x, dim=1)   [B, D]  (mean pool, NO CLS token)
        └──► patch_tokens  = x                [B, N, D]
             feature_maps  = reshape → [B, D, H/p, W/p]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor('#0f1117')

# ── Panel 1: 2D RoPE Diagram ─────────────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#0f1117')
ax1.set_xlim(-0.5, 10)
ax1.set_ylim(-0.5, 10)
ax1.axis('off')
ax1.set_title('2D Rotary Position Embedding (RoPE)', color='white', fontsize=11, fontweight='bold')

# Draw a query vector and show it being rotated
def vec_box(ax, x, y, w, h, items, color='#1a3a5c'):
    for i, item in enumerate(items):
        xi = x + i * w / len(items)
        p = FancyBboxPatch((xi, y), w/len(items)-0.05, h,
                            boxstyle='square,pad=0.02', facecolor=color,
                            edgecolor='#888', linewidth=0.8)
        ax.add_patch(p)
        ax.text(xi + w/len(items)/2 - 0.025, y + h/2, item, ha='center',
                va='center', color='white', fontsize=7.5, fontweight='bold')

# Original Q dim: [q₀,q₁ | q₂,q₃ | q₄,q₅ | q₆,q₇]
ax1.text(5, 9.5, 'Q vector [head_dim = 8 shown]', ha='center', color='white', fontsize=9)
items_q = ['q₀','q₁','q₂','q₃','q₄','q₅','q₆','q₇']
vec_box(ax1, 1, 8.5, 8, 0.6, items_q, '#1a3a5c')

# Brackets: row dims vs col dims
ax1.annotate('', xy=(1, 8.1), xytext=(5, 8.1),
             arrowprops=dict(arrowstyle='<->', color='#ff8c00', lw=1.5))
ax1.text(3, 7.85, 'Row position dims (head_dim/4)', ha='center', color='#ff8c00', fontsize=7.5)
ax1.annotate('', xy=(5, 8.1), xytext=(9, 8.1),
             arrowprops=dict(arrowstyle='<->', color='#44ff99', lw=1.5))
ax1.text(7, 7.85, 'Col position dims (head_dim/4)', ha='center', color='#44ff99', fontsize=7.5)

# Rotation formula
ax1.text(5, 7.4, 'Apply rotation (per pair of dims):', ha='center', color='#aaa', fontsize=8)

formulas = [
    ("q'₀ = q₀·cos(y·θ₀) − q₁·sin(y·θ₀)", '#ff8c00'),
    ("q'₁ = q₁·cos(y·θ₀) + q₀·sin(y·θ₀)", '#ff8c00'),
    ("q'₄ = q₄·cos(x·θ₀) − q₅·sin(x·θ₀)", '#44ff99'),
    ("q'₅ = q₅·cos(x·θ₀) + q₄·sin(x·θ₀)", '#44ff99'),
]
for i, (f, c) in enumerate(formulas):
    ax1.text(5, 6.9 - i*0.55, f, ha='center', va='center', color=c, fontsize=8, family='monospace')

# rotate_half explanation
ax1.text(5, 4.5, 'rotate_half: [a,b,c,d] → [−b,a,−d,c]', ha='center', color='white',
         fontsize=8.5, bbox=dict(boxstyle='round', facecolor='#2a2a4a', edgecolor='#4a9eff'))
ax1.text(5, 3.8, 'RoPE formula: q\' = q·cos + rotate_half(q)·sin',
         ha='center', color='white', fontsize=8.5,
         bbox=dict(boxstyle='round', facecolor='#1a4a2a', edgecolor='#44ff99'))

# Advantage
ax1.text(5, 2.8, 'Advantages over absolute pos embed:', ha='center', color='white', fontsize=8, style='italic')
advantages = [
    '• No separate pos embedding parameter',
    '• Generalises to unseen resolutions',
    '• Position info decays gracefully with distance',
    '• Relative positions matter: only dot-product q·k matters',
]
for i, a in enumerate(advantages):
    ax1.text(5, 2.3 - i*0.55, a, ha='center', color='#aaa', fontsize=7.5)

# ── Panel 2: SwiGLU FFN diagram ───────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0f1117')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 11)
ax2.axis('off')
ax2.set_title('SwiGLU FFN vs Standard GELU FFN', color='white', fontsize=11, fontweight='bold')

def box2(ax, x, y, w, h, txt, sub='', fc='#1a3a5c'):
    p = FancyBboxPatch((x-w/2, y-h/2), w, h, boxstyle='round,pad=0.05',
                        facecolor=fc, edgecolor='#4a9eff', linewidth=1.2)
    ax.add_patch(p)
    ax.text(x, y + (0.1 if sub else 0), txt, ha='center', va='center',
            color='white', fontsize=8.5, fontweight='bold')
    if sub:
        ax.text(x, y-0.28, sub, ha='center', va='center', color='#aaa', fontsize=7)

# Standard GELU MLP (left side)
ax2.text(2.5, 10.5, 'Standard GELU MLP', ha='center', color='#aaa', fontsize=9)
box2(ax2, 2.5, 9.8, 3.5, 0.55, 'x  [B, N, D]', '', '#333')
ax2.annotate('', xy=(2.5, 9.25), xytext=(2.5, 9.52),
             arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.2))
box2(ax2, 2.5, 9.0, 3.5, 0.45, 'Linear (D→4D)', '', '#1a3a5c')
ax2.annotate('', xy=(2.5, 8.48), xytext=(2.5, 8.78),
             arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.2))
box2(ax2, 2.5, 8.2, 3.5, 0.45, 'GELU(·)', '', '#1a4a2a')
ax2.annotate('', xy=(2.5, 7.68), xytext=(2.5, 7.98),
             arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.2))
box2(ax2, 2.5, 7.4, 3.5, 0.45, 'Linear (4D→D)', '', '#1a3a5c')
ax2.text(2.5, 6.8, '2 weight matrices\nParam: 2 × D × 4D = 8D²', ha='center',
         color='#aaa', fontsize=7.5)

# SwiGLU (right side)
ax2.text(7.5, 10.5, 'SwiGLU FFN', ha='center', color='#c44dff', fontsize=9, fontweight='bold')
box2(ax2, 7.5, 9.8, 3.5, 0.55, 'x  [B, N, D]', '', '#333')
# Two parallel branches
ax2.annotate('', xy=(6.2, 9.25), xytext=(7.1, 9.52),
             arrowprops=dict(arrowstyle='->', color='#c44dff', lw=1.2))
ax2.annotate('', xy=(8.8, 9.25), xytext=(7.9, 9.52),
             arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.2))
box2(ax2, 6.0, 8.95, 2.5, 0.45, 'W₁  (D→8/3D)', '', '#3a1a5a')
box2(ax2, 9.0, 8.95, 2.5, 0.45, 'W₂  (D→8/3D)', '', '#1a3a5c')
ax2.annotate('', xy=(6.0, 8.45), xytext=(6.0, 8.73),
             arrowprops=dict(arrowstyle='->', color='#c44dff', lw=1.2))
box2(ax2, 6.0, 8.2, 2.5, 0.45, 'SiLU(·)  gate', '', '#3a1a5a')
# Multiply
ax2.annotate('', xy=(7.5, 7.65), xytext=(6.5, 7.95),
             arrowprops=dict(arrowstyle='->', color='#c44dff', lw=1.2))
ax2.annotate('', xy=(7.5, 7.65), xytext=(8.8, 8.73),
             arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.2))
ax2.text(7.5, 7.5, '⊙', ha='center', va='center', color='white', fontsize=20, fontweight='bold')
ax2.annotate('', xy=(7.5, 7.05), xytext=(7.5, 7.35),
             arrowprops=dict(arrowstyle='->', color='#4a9eff', lw=1.2))
box2(ax2, 7.5, 6.75, 3.5, 0.45, 'W₃  (8/3D→D)', '', '#1a3a5c')
ax2.text(7.5, 6.1, '3 weight matrices\nParam: 3 × D × 8/3D = 8D²\n✓ Same params as GELU MLP!',
         ha='center', color='#44ff99', fontsize=7.5)

# ── Panel 3: Sub-LN explanation ───────────────────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor('#0f1117')
ax3.set_title('Sub-Layer Normalization (Sub-LN)', color='white', fontsize=11, fontweight='bold')

# Show logit explosion with and without Sub-LN
depths = np.arange(1, 33)
rng = np.random.default_rng(0)
logits_no_subnorm = 1.0 + np.cumsum(rng.normal(0, 0.15, 32) + 0.05)  # growing
logits_subnorm    = np.clip(1.0 + np.cumsum(rng.normal(0, 0.05, 32)), None, 3.0)

ax3.set_facecolor('#1a1a2e')
ax3.plot(depths, logits_no_subnorm, color='#ff4444', linewidth=2, label='Without Sub-LN')
ax3.plot(depths, logits_subnorm,    color='#44ff99', linewidth=2, label='With Sub-LN (EVA-02)')
ax3.set_xlabel('Transformer depth (blocks)', color='white')
ax3.set_ylabel('Max attention logit magnitude', color='white')
ax3.tick_params(colors='white')
ax3.legend(framealpha=0.3, labelcolor='white')
for sp in ax3.spines.values():
    sp.set_edgecolor('#444')

ax3.text(16, 2.5,
         'Sub-LN applies separate\nLayerNorm to Q and K\nbefore attention:\n\n'
         'Q = q_norm(Q)\nK = k_norm(K)\n\nNormalises magnitude of\nQ and K vectors,\n'
         'preventing logit explosion.',
         color='white', fontsize=8.5,
         bbox=dict(boxstyle='round', facecolor='#2a1a4a', edgecolor='#c44dff', alpha=0.9))

plt.tight_layout()
plt.savefig('eva02_architecture.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('EVA-02 architecture diagram saved.')

## Key Innovations

### 1. 2D Rotary Position Embedding (RoPE)

Standard ViT adds learned absolute positional embeddings to token features before the first block. RoPE instead **encodes position into the rotation of Q and K vectors** inside each attention block.

For a 1D sequence at position $m$, RoPE rotates each consecutive pair $(x_{2i}, x_{2i+1})$:
$$\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

where $\theta_i = 1/10000^{2i/d}$ (same as Transformer sinusoidal frequencies).

For 2D images, EVA-02 uses:
- The **first half** of head dimensions → row-position RoPE (using $y$-coordinate)
- The **second half** of head dimensions → column-position RoPE (using $x$-coordinate)

**Key property:** RoPE is applied to Q and K only. After rotation,
$$q_m^\top k_n = f(q, m)^\top f(k, n)$$
only depends on the **relative** position $(m - n)$, not absolute positions. This means RoPE implicitly computes relative position attention without a separate relative position bias table.

### 2. SwiGLU Feed-Forward Network

Standard ViT MLP: $\text{MLP}(x) = W_2 \cdot \text{GELU}(W_1 x)$ — 2 weight matrices, hidden dim $= 4D$

SwiGLU (Shazeer, 2020):
$$\text{SwiGLU}(x) = W_3 \cdot \bigl(\text{SiLU}(W_1 x) \odot W_2 x\bigr)$$

- 3 weight matrices instead of 2
- Hidden dim scaled to $\lceil 8D/3 / 256 \rceil \times 256 \approx 2.67D$
- Total params: $3 \times D \times \frac{8D}{3} = 8D^2$ — same as GELU MLP ($2 \times D \times 4D = 8D^2$) ✓
- The gating mechanism ($\text{SiLU}(W_1 x) \odot W_2 x$) provides input-dependent selection, empirically outperforming non-gated activations.

### 3. Sub-Layer Normalization (Sub-LN)

Applies separate `LayerNorm` to Q and K **before** computing attention:
```
Q, K, V = in_proj(x)
Q = q_norm(Q)    # LayerNorm per head
K = k_norm(K)    # LayerNorm per head
Attn = softmax(Q·Kᵀ / √d) · V
```

Without Sub-LN, at large scale (large $D$, many heads), Q and K vectors can develop large magnitudes that saturate the softmax (all attention on one token). Sub-LN normalises their magnitude independently of the pre-block LayerNorm.

### 4. No CLS Token — Mean Pooling

EVA-02 uses mean pooling over all patch tokens rather than a special CLS token:
$$z = \frac{1}{N} \sum_{i=1}^{N} x_i$$

This makes all patches equal contributors to the global descriptor and removes the special-token asymmetry.

## Module Walkthrough

### 2D RoPE — Pre-computation
```python
def _build_rope_freqs(grid_size, head_dim, base=10000.0):
    half = head_dim // 4  # row: head_dim/4 dims, col: head_dim/4 dims
    theta = 1.0 / (base ** (arange(0, half) / half))  # frequency bands
    # Grid (row, col) positions:
    py = row_positions.reshape(-1)  # [N]
    px = col_positions.reshape(-1)  # [N]
    ang_y = py[:, None] * theta[None, :]    # [N, half]
    ang_x = px[:, None] * theta[None, :]    # [N, half]
    ang = cat([ang_y, ang_x], dim=1)        # [N, head_dim/2]
    ang = ang.repeat_interleave(2, dim=1)   # [N, head_dim]
    return ang.cos(), ang.sin()             # both [N, head_dim]
```

### _apply_rope — Rotation of Q and K
```python
def _rotate_half(x):
    # [a, b, c, d] → [−b, a, −d, c]
    x1, x2 = x[..., ::2], x[..., 1::2]   # split even/odd
    return stack([-x2, x1], dim=-1).flatten(-2)

def _apply_rope(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)    # [1, 1, N, d]
    sin = sin.unsqueeze(0).unsqueeze(0)
    q_rot = q * cos + _rotate_half(q) * sin
    k_rot = k * cos + _rotate_half(k) * sin
    return q_rot, k_rot
```

### SwiGLUFFN
```python
class SwiGLUFFN(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0):
        hidden = ceil(dim * mlp_ratio * 2/3 / 256) * 256  # ≈ 8/3 × dim
        self.w1 = nn.Linear(dim, hidden, bias=False)  # gate
        self.w2 = nn.Linear(dim, hidden, bias=False)  # value
        self.w3 = nn.Linear(hidden, dim, bias=False)  # project back

    def forward(self, x):
        gate  = F.silu(self.w1(x))     # SiLU activation = x·σ(x)
        value = self.w2(x)
        return self.w3(gate * value)   # element-wise gate
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
from hale_core.nn.vision import build_vision_backbone

backbone = build_vision_backbone('eva_02_small', img_size=224)
backbone.eval()
print(f'Parameters:   {backbone.num_parameters():,}')
print(f'Embed dim:    {backbone.embed_dim}')
print(f'Patch size:   {backbone.patch_size}')
print(f'RoPE cos buf: {backbone.rope_cos.shape}')  # [N, head_dim]

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print(f'\npooled_output:  {out.pooled_output.shape}')   # [1, 384]
print(f'patch_tokens:   {out.patch_tokens.shape}')      # [1, 196, 384]
print(f'feature_maps:   {out.feature_maps[0].shape}')   # [1, 384, 14, 14]

# Visualise RoPE frequency patterns
cos_freqs = backbone.rope_cos.numpy()  # [N, head_dim]
N = cos_freqs.shape[0]
G = int(N**0.5)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('EVA-02 RoPE cos Frequencies — First 8 Dimension Pairs', color='white', fontsize=12)

for i, ax in enumerate(axes.flat):
    ax.set_facecolor('#0f1117')
    if i < 8:
        freq_map = cos_freqs[:, i*2].reshape(G, G)
        im = ax.imshow(freq_map, cmap='coolwarm', interpolation='nearest', vmin=-1, vmax=1)
        ax.set_title(f'dim {i*2}', color='white', fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046)
    ax.tick_params(colors='white')
    for sp in ax.spines.values():
        sp.set_edgecolor('#333')

plt.tight_layout()
plt.savefig('eva02_rope_freqs.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Low-freq dims (left): gradual spatial variation = global position')
print('High-freq dims (right): rapid variation = fine-grained position discrimination')

## References

### Primary
1. Y. Fang et al., "EVA-02: A Visual Representation Powerhouse Reconstructed from EVA," *arXiv:2303.11331*, 2023. [arXiv:2303.11331](https://arxiv.org/abs/2303.11331)
2. Y. Fang et al., "EVA: Exploring the Limits of Masked Visual Representation Learning at Scale," *CVPR 2023*. [arXiv:2211.07636](https://arxiv.org/abs/2211.07636) — EVA v1

### Related Works
3. N. Shazeer, "GLU Variants Improve Transformer," *arXiv:2002.05202*, 2020. [arXiv:2002.05202](https://arxiv.org/abs/2002.05202) — SwiGLU
4. J. Su et al., "RoFormer: Enhanced Transformer with Rotary Position Embedding," *Neurocomputing*, 2024. [arXiv:2104.09864](https://arxiv.org/abs/2104.09864) — RoPE
5. A. Henry et al., "Query-Key Normalization for Transformers," *EMNLP Findings 2020*. [arXiv:2010.04245](https://arxiv.org/abs/2010.04245) — QK Normalization
6. A. Radford et al., "Learning Transferable Visual Models From Natural Language Supervision," *ICML 2021*. [arXiv:2103.00020](https://arxiv.org/abs/2103.00020) — CLIP (EVA-02 trains with CLIP alignment)
7. A. Dosovitskiy et al., "An Image is Worth 16×16 Words," *ICLR 2021*. [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)